# Pitcher strikeout exploratory analysis

This notebook reads the canonical Level 1 `pitcher_games.parquet` artifact.
Reusable stabilization and reliability logic lives in `mlb_props.reliability`;
model-ready features are produced separately by pipeline Levels 2 and 3.

**Leakage reminder:** `PA`, `K`, and other same-game outcomes are valid here for
EDA and reliability analysis only. They must never enter the pregame feature set.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

# --- Plot rendering toggle (keeps this .ipynb small so it doesn't bloat AI context) ---
# With the non-interactive "Agg" backend, figures are still written to the Graphs/
# folder by plt.savefig(...) but are NOT embedded inline, so no heavy base64 image
# outputs get stored in the notebook. plt.show() becomes a harmless no-op.
# To view plots inline instead: comment out the next line, then restart the kernel.
plt.switch_backend("Agg")

# Locate repo root (folder containing src/mlb_props) to import the package.
_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "src" / "mlb_props").exists():
    _root = _root.parent
if (_root / "src").exists() and str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from mlb_props.config import PITCHER_GAMES_PATH
from mlb_props import reliability as rel

# Graphs go next to this notebook, so renaming the folder never breaks the path.
try:
    NB_DIR = Path(__vsc_ipynb_file__).resolve().parent  # Cursor/VSCode
except NameError:
    NB_DIR = Path.cwd()
GRAPHS = NB_DIR / "Graphs"
GRAPHS.mkdir(parents=True, exist_ok=True)
print("Data:", PITCHER_GAMES_PATH)
print("Graphs ->", GRAPHS)

In [ ]:
games = pl.read_parquet(PITCHER_GAMES_PATH)
df = games.to_pandas()
print("rows, cols:", df.shape)
df.head()

## 1. Target distribution
Raw K, K/PA rate, and log1p(K).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["K"], bins=30); axes[0].set_title("K raw")
axes[1].hist(df["k_rate"], bins=30); axes[1].set_title("K/PA rate")
axes[2].hist(np.log1p(df["K"]), bins=30); axes[2].set_title("log1p(K)")
plt.tight_layout(); plt.savefig(GRAPHS / "k_distribution.png", dpi=120); plt.show()

## 2. Correlation with K

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
corr = df[numeric_cols].corr()["K"].sort_values(ascending=False)
print("Top +correlated with K:"); print(corr.head(20))
print("\nMost -correlated with K:"); print(corr.tail(10))

top_features = corr.abs().sort_values(ascending=False).head(30).index
plt.figure(figsize=(16, 14))
sns.heatmap(df[top_features].corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Top 30 Feature Correlation Matrix")
plt.tight_layout(); plt.savefig(GRAPHS / "corr_heatmap.png", dpi=120); plt.show()

## 3. Skew audit
Flag features that may benefit from a log/transform.

In [ ]:
skew_report = {}
for col in numeric_cols:
    if df[col].notna().sum() > 100:
        skew_report[col] = skew(df[col].dropna())
skew_s = pd.Series(skew_report).sort_values(ascending=False)
print("High positive skew (consider log):"); print(skew_s[skew_s > 1])
print("\nHigh negative skew:"); print(skew_s[skew_s < -1])

## 4. Physics features vs K/PA
Scatter with a linear fit for a few key pitch-shape features.

In [ ]:
physics_features = [
    "ff_vaa", "ff_ivb", "ff_velo", "ff_spinrate",
    "sl_vaa", "ch_vaa", "cu_vaa",
    "ff_usage_vR", "ff_usage_vL",
]
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, feat in zip(axes.flatten(), physics_features):
    if feat not in df.columns:
        ax.set_visible(False); continue
    sub = df[["k_rate", feat]].dropna()
    ax.scatter(sub[feat], sub["k_rate"], alpha=0.3, s=10)
    ax.set_xlabel(feat); ax.set_ylabel("K/PA")
    if len(sub) > 2:
        m, b = np.polyfit(sub[feat], sub["k_rate"], 1)
        ax.plot(sub[feat], m * sub[feat] + b, color="red", linewidth=1)
plt.tight_layout(); plt.savefig(GRAPHS / "scatter_physics.png", dpi=120); plt.show()

## 5. Null audit
Anything above ~10% null is worth flagging for imputation strategy.

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False)
print("Features >10% null:"); print(null_pct[null_pct > 0.1])
ax = null_pct[null_pct > 0].plot(kind="bar", figsize=(16, 5))
ax.set_title("Null Rate per Feature")
plt.tight_layout(); plt.savefig(GRAPHS / "null_audit.png", dpi=120); plt.show()

## 6. K vs PA (leakage awareness)

`PA` scales `K` almost mechanically, which is exactly why same-game `PA` cannot be
a pregame feature. This plot is a reminder of that relationship, not a feature.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["PA"], df["K"], alpha=0.3, s=10)
plt.xlabel("PA"); plt.ylabel("K"); plt.title("K vs PA (outcome, not a feature)")
plt.tight_layout(); plt.savefig(GRAPHS / "k_vs_pa.png", dpi=120); plt.show()

## 7. Stabilization curves

Split-half reliability as a function of window size (starts per half). The dashed
line marks r = 0.70. Uses `mlb_props.reliability.stabilization_curve`.

In [ ]:
game_range = list(range(1, 40))

# Single grouped pass with cumulative-sum window slicing (fast).
rel_df = rel.stabilization_curve(df, windows=game_range)

# Figure 1: count/rate stats
fig, ax = plt.subplots(figsize=(14, 6))
for stat in rel.RATE_STATS:
    if stat in rel_df.columns:
        ax.plot(game_range, rel_df[stat], label=stat, linewidth=2)
ax.axhline(0.70, color="black", linestyle="--", linewidth=1, label="r=0.70")
ax.set_xlabel("Starts per half"); ax.set_ylabel("Split-half r")
ax.set_title("Count/Rate Stat Stabilization"); ax.legend(ncol=3, fontsize=8)
plt.tight_layout(); plt.savefig(GRAPHS / "stabilization_counts.png", dpi=120); plt.show()

# Figure 2: pitch physics by pitch type
pitch_groups = {p: [f"{p.lower()}_{s}" for s in ["velo", "spinrate", "ivb", "hb", "vaa"]]
                for p in ["FF", "SI", "SL", "FC", "ST", "CU", "CH"]}
fig, axes = plt.subplots(2, 4, figsize=(22, 10)); axes = axes.flatten()
for ax, (pitch, cols) in zip(axes, pitch_groups.items()):
    for col in cols:
        if col in rel_df.columns:
            ax.plot(game_range, rel_df[col], label=col.split("_", 1)[1], linewidth=1.5)
    ax.axhline(0.70, color="black", linestyle="--", linewidth=1)
    ax.set_title(pitch); ax.set_ylim(0, 1)
    ax.set_xlabel("Starts per half"); ax.set_ylabel("r"); ax.legend(fontsize=7)
axes[-1].set_visible(False)
plt.suptitle("Pitch-Level Stat Stabilization by Pitch Type", fontsize=14)
plt.tight_layout(); plt.savefig(GRAPHS / "stabilization_pitch.png", dpi=120); plt.show()

# Summary: starts-per-half to reach r=0.70
points = {}
for stat in rel_df.columns:
    series = rel_df[stat].dropna()
    crossed = series[series >= 0.70]
    points[stat] = int(crossed.index[0]) if len(crossed) else ">39"
print("Stabilization points (starts per half to reach r=0.70):")
print(pd.Series(points).sort_values(key=lambda s: s.map(lambda v: 999 if v == ">39" else v)).to_string())

## 8. Enhanced reliability table

Split-half Pearson/Spearman, ICC(2,1) (if `pingouin` is installed), year-over-year
correlation, standard error of measurement, and regression slope for each stat.
Uses `mlb_props.reliability.reliability_table`.

In [ ]:
metrics_df = rel.reliability_table(df, n_games_per_half=10)
print(f"Enhanced reliability (n=10 starts per half), sorted by YoY r:\n")
cols = ["stat", "pearson_r", "spearman_rho", "icc2", "yoy_r", "sem", "beta1", "feature_tier"]
print(metrics_df[cols].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## 9. Reliability diagnostics

- **Pearson/Spearman gap > 0.10** -> correlation is outlier-driven.
- **High r but low slope** -> strong regression to the mean.
- **Feature tiers** by year-over-year reliability.

In [ ]:
gap = (metrics_df["pearson_r"] - metrics_df["spearman_rho"]).abs()
flagged = metrics_df.assign(gap=gap).query("gap > 0.10").sort_values("gap", ascending=False)
print("Outlier-driven stats (|Pearson - Spearman| > 0.10):")
print(flagged[["stat", "pearson_r", "spearman_rho", "gap"]].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

low_beta = metrics_df.query("pearson_r >= 0.50 and beta1 < 0.60").sort_values("beta1")
print("\nHigh Pearson r but low slope (strong regression-to-mean):")
print(low_beta[["stat", "pearson_r", "beta1", "yoy_r"]].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nFeature tiers by YoY reliability:")
for tier in ["Tier 1 - stable", "Tier 2 - moderate", "Tier 3 - noisy", "Unknown"]:
    subset = metrics_df[metrics_df["feature_tier"] == tier]["stat"].tolist()
    print(f"{tier} ({len(subset)}): " + ", ".join(subset))